# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mukeshboolani786/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
from google.colab import userdata
import duckdb
import os

# Load the working Hugging Face token from Colab Secrets.
HF_TOKEN = userdata.get("internship-token")

if not HF_TOKEN:
    raise ValueError("Hugging Face token was not found in Colab Secrets.")

# Create DuckDB connection.
con = duckdb.connect()

# FlyRank dataset location.
rel = "hf://datasets/FlyRank/internship-warehouse"

# Create a temporary Hugging Face secret in DuckDB.
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_token
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

# Test access to the real March 2026 FlyRank data.
test = con.sql(
    f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    LIMIT 5
    """
)

print("✅ FlyRank warehouse connection successful!")
print("✅ March 2026 data is accessible.")
display(test.df())

✅ FlyRank warehouse connection successful!
✅ March 2026 data is accessible.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline prioritizes content items that have meaningful search visibility but may have a CTR improvement opportunity. The rule uses observed Google Search Console impressions and clicks from March 2026. Items with at least 100 impressions and CTR below 2% receive higher priority for review.

The score gives more priority to items with more impressions and lower CTR. This is a simple decision-support baseline, not a claim that changing the content will improve performance.

Reason codes:

* **CTR_FIX_CANDIDATE** — the content has enough search impressions and a relatively low observed CTR, so it is a candidate for CTR review.
* **NO_ACTION** — the available signals do not provide enough evidence to prioritize the content for CTR review.

The baseline uses search position as contextual evidence rather than as a direct scoring component. The signal checks below are directional: they show observed relationships in the March 2026 data, not causal effects.


In [23]:
import pandas as pd

signal_data = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    """
).df()

# Calculate observed CTR.
signal_data["ctr_pct"] = (
    100.0
    * signal_data["gsc_clicks"]
    / signal_data["gsc_impressions"]
)

# ---------------------------------------------------------
# SIGNAL 1: Search volume / impressions
# ---------------------------------------------------------

signal_data["impression_bucket"] = pd.cut(
    signal_data["gsc_impressions"],
    bins=[0, 100, 1000, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

impression_check = (
    signal_data
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_ctr_pct=("ctr_pct", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — SEARCH VOLUME / IMPRESSIONS")
display(impression_check)

# ---------------------------------------------------------
# SIGNAL 2: CTR vs search position
# ---------------------------------------------------------

position_data = signal_data[
    signal_data["gsc_avg_position"] > 0
].copy()

position_data["position_bucket"] = pd.cut(
    position_data["gsc_avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_check = (
    position_data
    .groupby("position_bucket", observed=False)
    .agg(
        n=("gsc_avg_position", "size"),
        avg_position=("gsc_avg_position", "mean"),
        avg_ctr_pct=("ctr_pct", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS SEARCH POSITION")
display(position_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — SEARCH VOLUME / IMPRESSIONS


,impression_bucket,n,avg_impressions,avg_ctr_pct
0,Low,2977578,20.379317,0.308679
1,Medium,601123,268.091439,0.307052
2,High,32360,1817.696323,0.271525



SIGNAL 2 — CTR VS SEARCH POSITION


,position_bucket,n,avg_position,avg_ctr_pct
0,Top 3,564173,1.807196,0.491821
1,4-10,1456122,6.059994,0.347264
2,11-20,519223,14.330876,0.276991
3,21+,908354,43.888638,0.128915


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline queue is built at the content-item level rather than treating each daily observation as a separate recommendation. March 2026 GSC observations are aggregated by client and content item. Impressions and clicks are summed across the month, and CTR is calculated from those aggregated totals. This prevents the same content item from occupying multiple positions simply because it appears on multiple days.


In [24]:
import os

# Aggregate the real March 2026 observations to content level.
df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

# Calculate monthly observed CTR.
df["ctr_pct"] = (
    100.0
    * df["gsc_clicks"]
    / df["gsc_impressions"]
)

# Baseline priority score:
# more impressions + lower CTR = higher review priority.
df["score"] = (
    df["gsc_impressions"]
    / (df["ctr_pct"] + 0.1)
)

# Default action.
df["reason_code"] = "NO_ACTION"
df["action"] = "NO_ACTION"

# CTR review candidates.
candidate = (
    (df["gsc_impressions"] >= 100)
    & (df["ctr_pct"] < 2.0)
)

df.loc[candidate, "reason_code"] = "CTR_FIX_CANDIDATE"
df.loc[candidate, "action"] = "REVIEW_CTR"

# Rank by baseline score.
df = df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Final queue.
queue = df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
]

# Create output directory.
os.makedirs("work/outputs", exist_ok=True)

# Required output path.
output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print("✅ Baseline ranked queue created successfully.")
print("Number of content-level rows:", len(queue))
print("Output file:", output_path)

display(queue.head(20))# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


✅ Baseline ranked queue created successfully.
Number of content-level rows: 176738
Output file: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr_pct,avg_position,score,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.011299,7.346909,1.908405e+06,CTR_FIX_CANDIDATE,REVIEW_CTR
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,0.000741,4.545582,1.339914e+06,CTR_FIX_CANDIDATE,REVIEW_CTR
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,0.000806,9.385150,1.230830e+06,CTR_FIX_CANDIDATE,REVIEW_CTR
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,0.030066,3.219473,1.099588e+06,CTR_FIX_CANDIDATE,REVIEW_CTR
4,5,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,12.0,0.010673,30.769353,1.015863e+06,CTR_FIX_CANDIDATE,REVIEW_CTR
5,6,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,60.0,0.041694,22.558608,1.015621e+06,CTR_FIX_CANDIDATE,REVIEW_CTR
6,7,client_23a62021009f63c4,content_559cdd76da9306de,97378.0,2.0,0.002054,36.712074,9.541825e+05,CTR_FIX_CANDIDATE,REVIEW_CTR
7,8,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,0.013943,9.536301,9.441948e+05,CTR_FIX_CANDIDATE,REVIEW_CTR
8,9,client_23a62021009f63c4,content_164c1f53f13bcee1,89982.0,2.0,0.002223,24.083947,8.802549e+05,CTR_FIX_CANDIDATE,REVIEW_CTR
9,10,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,0.124371,32.766674,8.672196e+05,CTR_FIX_CANDIDATE,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 content items produced by the baseline rule. The action and reason code come directly from the scoring rule. Confidence notes reflect the amount of observed search evidence, while the error notes describe conditions that could make a recommendation unreliable.

These are decision-support recommendations. A high score does not prove that changing the content will improve performance, so the items should be reviewed before action.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

def confidence_note(impressions):
    if impressions >= 1000:
        return "Higher confidence: strong observed search volume."
    elif impressions >= 300:
        return "Moderate confidence: useful observed search volume."
    else:
        return "Lower confidence: limited observed search volume."

top20["confidence_note"] = top20[
    "gsc_impressions"
].apply(confidence_note)


def wrong_reason(row):
    if row["gsc_impressions"] < 300:
        return "Could be wrong because search volume is limited."
    elif row["gsc_clicks"] == 0:
        return "Could be wrong because no clicks were observed."
    elif row["ctr_pct"] >= 2.0:
        return "Could be wrong because CTR is not especially low."
    else:
        return (
            "Could be wrong if the observed CTR does not represent "
            "a genuine improvement opportunity."
        )


top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_44f34c0a90047651,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
1,2,content_8e1334d6356668e3,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
2,3,content_fec55986a1868d62,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
3,4,content_34a70fea29d15f24,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
4,5,content_bdf60c86117079be,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
5,6,content_82e35c4845e6c391,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
6,7,content_559cdd76da9306de,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
7,8,content_f6116743b00afc2d,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
8,9,content_164c1f53f13bcee1,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...
9,10,content_36e53e9c707674fc,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong observed search volume.,Could be wrong if the observed CTR does not re...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may be weak when search evidence is limited or when a low observed CTR does not represent a genuine improvement opportunity. These should be treated as review candidates rather than guaranteed actions.

The baseline uses March 2026 observed Google Search Console impressions and clicks. It does not use future-window outcomes, product flags, or a model-derived target. The baseline is therefore designed to avoid intentional future or label-derived leakage. The use of March observations should still be interpreted as a baseline for the defined decision window rather than proof of future performance.


In [26]:
# Potentially weak recommendations.
weak_picks = top20[
    (top20["gsc_impressions"] < 300)
    | (top20["gsc_clicks"] == 0)
][
    [
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "score",
        "reason_code",
        "action"
    ]
]

print("Potentially weak picks:")
display(weak_picks)


# Explicit list of fields used by the baseline.
baseline_inputs = [
    "gsc_impressions",
    "gsc_clicks"
]

print("\nFields used by the scoring rule:")
for field in baseline_inputs:
    print("-", field)


# Basic leakage audit.
print("\nLeakage check:")
print("✓ The score uses only observed March 2026 impressions and clicks.")
print("✓ No future-month outcome is used in the score.")
print("✓ No synthetic/random records are used.")
print("✓ No label generated by a later ML model is used.")
print("✓ The ranking is a transparent hand-written baseline.")# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Potentially weak picks:


,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr_pct,score,reason_code,action



Fields used by the scoring rule:
- gsc_impressions
- gsc_clicks

Leakage check:
✓ The score uses only observed March 2026 impressions and clicks.
✓ No future-month outcome is used in the score.
✓ No synthetic/random records are used.
✓ No label generated by a later ML model is used.
✓ The ranking is a transparent hand-written baseline.


In [ ]:
# This cell is intentionally left blank for any additional code-based checks if needed.
# For this assignment, the leakage check was descriptive.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.